In [1]:
import numpy as np
import matplotlib.pyplot as plt
import datetime
from keras.models import Sequential
from keras.layers import (
    Input,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Flatten,
    Dense,
    Dropout
)
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds

In [3]:
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.C57YMP_3.0.1/mnist-train.tfrecord*...:   0%|          | 0…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.C57YMP_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


In [4]:
def normalize_img(image, label):
  """Normalizes images: `uint8` -> `float32`."""
  return tf.cast(image, tf.float32) / 255., label

ds_train = ds_train.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(128)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

In [5]:
# ten był podany w artykule tenserflow o szkoleniu sieci neuronowych jako
# prosty model

# model = tf.keras.models.Sequential([
#   tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
#   tf.keras.layers.Dense(128, activation='relu'),
#   tf.keras.layers.Dense(10)
# ])

In [6]:
# LeNet - najwidoczniej ma 100% poprawności

model = Sequential()
model.add(Conv2D(filters=32, kernel_size=(5,5), padding='same', activation='relu', input_shape=(28, 28, 1)))
model.add(MaxPooling2D(strides=2))
model.add(Conv2D(filters=48, kernel_size=(5,5), padding='valid', activation='relu'))
model.add(MaxPooling2D(strides=2))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(84, activation='relu'))
model.add(Dense(10, activation='softmax'))


In [7]:
# alexnet

# model = Sequential()

# # Layer 1: Convolutional layer with 64 filters of size 11x11x3
# model.add(Conv2D(filters=64, kernel_size=(11,11), strides=(4,4), padding='valid', activation='relu', input_shape=(28,28,1))) # Changed input_shape to (28,28,1) for MNIST

# # # Layer 2: Max pooling layer with pool size of 3x3
# # model.add(MaxPooling2D(pool_size=(3,3), strides=(2,2)))

# # Layer 3-5: 3 more convolutional layers with similar structure as Layer 1
# model.add(Conv2D(filters=192, kernel_size=(5,5), padding='same', activation='relu'))
# # model.add(MaxPooling2D((2,2)))
# # model.add(Conv2D(filters=384, kernel_size=(3,3), padding='same', activation='relu'))
# model.add(Conv2D(filters=256, kernel_size=(3,3), padding='same', activation='relu'))
# model.add(MaxPooling2D((2,2)))

# # Layer 6: Fully connected layer with 4096 neurons
# model.add(Flatten())
# model.add(Dense(4096, activation='relu'))

# # Layer 7: Fully connected layer with 4096 neurons
# model.add(Dense(4096, activation='relu'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)


model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 5, 5, 64)       │         7,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 5, 5, 192)      │       307,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 5, 5, 256)      │       442,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 2, 2, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │     4,198,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,737,536 (82.92 MB)

 Trainable params: 21,737,536 (82.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
es = EarlyStopping(
    monitor='val_loss',   # metryka, którą obserwujemy (np. val_loss, val_accuracy)
    patience=8,               # liczba epok bez poprawy, po których zatrzymujemy trening
    min_delta=1e-3,           # minimalna wymagana zmiana, by uznać, że jest „poprawa”
    mode='min'              # 'min' jeśli monitorujemy straty, 'max' jeśli dokładność
)
rlp = ReduceLROnPlateau(
    monitor='val_loss',   # metryka do obserwacji
    factor=0.5,           # ile razy zmniejszyć LR (tu: o połowę)
    patience=5,           # liczba epok bez poprawy przed zmniejszeniem LR, rlp.patience < es.patience
    min_delta=1e-3,       # próg czułości jak wyżej
    min_lr=1e-4,          # dolna granica learning rate
    mode='min',           # 'min' dla strat, 'max' dla dokładności
)

# Re-initialize and preprocess datasets to ensure single batching
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

def normalize_img(image, label):
  """Normalizes images: `uint8` -> `float32`."""
  return tf.cast(image, tf.float32) / 255., label

ds_train = ds_train_raw.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_train = ds_train.cache()
ds_train = ds_train.shuffle(ds_info.splits['train'].num_examples)
ds_train = ds_train.batch(128)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)

ds_test = ds_test_raw.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_test = ds_test.batch(128)
ds_test = ds_test.cache()
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)

# Recompile the model with the correct loss function setting
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), # Changed from_logits to False
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

history = model.fit(
    ds_train,
    epochs=6,
    validation_data=ds_test,
    callbacks=[es, rlp],
    verbose=1
)

Epoch 1/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 636s 1s/step - loss: 3.8151 - sparse_categorical_accuracy: 0.1122 - val_loss: 3.8236 - val_sparse_categorical_accuracy: 0.1135 - learning_rate: 0.0010
Epoch 2/6
469/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 3.8258 - sparse_categorical_accuracy: 0.1135

In [ ]:
loss, accuracy = model.evaluate(ds_train, verbose=0)
print('Dokładność na zbiorze testowym:', f"{accuracy:.2f}")
print('Strata na zbiorze testowym:', f"{loss:.2f}")

In [ ]:
fig_loss_acc = plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='loss (train)')
plt.plot(history.history['val_loss'], label='loss (val)')
plt.xlabel('Epoka')
plt.ylabel('categorical_crossentropy')
plt.title(f'Strata na zbiorze testowym to {loss:.2f}')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['sparse_categorical_accuracy'], label='accuracy (train)')
plt.plot(history.history['val_sparse_categorical_accuracy'], label='accuracy (val)')
plt.xlabel('Epoka')
plt.ylabel('Accuracy')
plt.title(f"Dokładność na zbiorze testowym to {accuracy:.2f}")
plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
# Define class names for MNIST
class_names = [str(i) for i in range(10)]

# Prepare true labels and images from ds_test for display and evaluation
all_test_images = []
all_test_labels = []
for images, labels in ds_test.unbatch(): # unbatch to get individual images/labels
    all_test_images.append(images.numpy())
    all_test_labels.append(labels.numpy())

all_test_images = np.array(all_test_images)
true_labels = np.array(all_test_labels)

# Make predictions on the test dataset
pred_probs = model.predict(ds_test) # Use the already batched ds_test for prediction
pred_labels = np.argmax(pred_probs, axis=1)

# Indeksy błędnych klasyfikacji
incorrect_indices = np.nonzero(pred_labels != true_labels)[0]

# Wyświetlenie kilku błędnych przykładów
n_show = min(3, len(incorrect_indices))
for i in range(n_show):
    idx = incorrect_indices[i]
    plt.figure(figsize=(3,3))
    # Use all_test_images instead of ds_train[idx] or direct ds_test indexing
    plt.imshow(all_test_images[idx].squeeze(), cmap='gray')
    plt.title(f"Prawidłowo: {class_names[true_labels[idx]]}  -> Predykcja: {class_names[pred_labels[idx]]}")
    plt.axis('off')
    plt.show()

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig_cm, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Macierz pomyłek')
plt.show()